# DS4 Stochastic Processes (StoP) -- Stage 4 Step 1

**Notebook:** `53_ds4_stochastic_processes.ipynb`  
**Model family:** Stochastic Processes (StoP)  
**Dataset:** DS4 -- Large Naturalistic OneSec Users (N=1,039)  
**Stage:** 4 . Step 1 -- Theoretical and Conceptual Modeling  
**Prerequisites:** None -- first DS4 model. InT(52)/DyS(55)/StM(51) depend on this notebook's output; NeT(54) does not.  
**Blueprint:** `outputs/reports/stage4/step1_modeling/ds4_stop_blueprint.md`

---

## Goal

Fit an **Ornstein-Uhlenbeck (OU) process** -- `dX_t = theta(mu - X_t)dt + sigma dW_t` -- to each DS4 participant's daily intervention-override rate. This is DS4's central methodological upgrade over DS5, which used a constant-drift SDE because its 7-45 day windows could not identify a mean-reversion parameter. DS4's windows (up to 897 days) make theta potentially identifiable for the first time in this project.

## Not the goal

- No condition comparison -- DS4 has no experimental condition.
- No claim about which behavioral interpretation (habit vs. measurement noise) the fast-theta finding reflects -- both readings are reported, see the blueprint Section 4 and the interpretation cell at the end of this notebook.
- No InT/DyS/StM computation -- those are separate notebooks that consume this one's output.

## Core claim

> This notebook fits a genuinely different process from DS5's StoP model, not a port of it. The estimation method (closed-form conditional MLE via AR(1) regression on consecutive-day pairs) is exact for the discrete-time OU transition at fixed delta-t=1 day -- not an approximation. All state-variable and eligibility choices are grounded in DS4's own confirmed structure (event-class filtering, inverted setup-time columns, no condition variable), not carried over from DS5.

## Output

```
outputs/model_outputs/ds4_stochastic_processes/
  participant_parameters.csv
  participant_parameters_weekly_sensitivity.csv
  ds4_stop_summary.csv
  model_run_summary.md
  figures/
```


## FLAG decisions

| Flag | Decision | Justification |
|---|---|---|
| FLAG 1 -- process type | OU: `dX_t=theta(mu-X_t)dt+sigma dW_t` | Long DS4 windows make theta identifiable, unlike DS5 |
| FLAG 2 -- estimation method | Closed-form conditional MLE via AR(1) OLS regression on consecutive-day pairs | Exact for the discrete-time OU transition at fixed delta-t=1; 96.16% of day-to-day pairs in DS4 are consecutive |
| FLAG 3 -- MIN_DAYS threshold | 30 distinct daily X_t observations, and >=10 usable consecutive-day pairs | Verified: 786/1,039 (75.6%) meet the 30-day threshold; 785 retain >=10 consecutive pairs |
| FLAG 4 -- invalid-fit handling | phi_hat outside (0,1) -> NaN theta/sigma/tau, excluded not imputed | Verified: 641/785 (81.7%) produce a valid fit |
| FLAG 5 -- daily event-count floor | None by default | Robustness-checked in cell 12: median theta_hat_OU is stable (1.587/1.464/1.378) across 1/3/5-event floors |

See the blueprint for full justification of each row.


## DS4 structural cautions

- **Event class:** state variable uses `intervention_event_class == 'intervention_event'` rows only, outcome in `{opened, dismissed}`. Never `rate_opened_all` -- it mixes friction and non-friction events (see `data_docs/modeling_cautions.md` SS6 DS4).
- **Time axis:** `timestamp_utc` for calendar-day binning. Never `secondsSinceStudySetup`/`weeksSinceStudySetup` -- confirmed entirely non-positive, encoding a backward-counting per-participant anchor, not forward time (Stage 0 audit SS6).
- **No experimental condition:** DS4 is observational. All results below describe within-participant and cross-participant heterogeneity, never a condition effect.


## Theory: the Ornstein-Uhlenbeck process

$$dX_t = \theta(\mu - X_t)\,dt + \sigma\, dW_t$$

Stationary distribution (exact): $P_\infty(x) = \mathcal{N}(\mu, \sigma^2/2\theta)$.

At fixed $\Delta t$, the exact discrete-time transition is Gaussian and linear in $X_t$ -- an AR(1) process:

$$X_{t+1} \mid X_t \sim \mathcal{N}\big(\mu + (X_t-\mu)e^{-\theta\Delta t},\; \sigma^2(1-e^{-2\theta\Delta t})/(2\theta)\big)$$

For $\Delta t = 1$ day (consecutive days), this is exactly $X_{t+1} = c + \phi X_t + \varepsilon_t$ with $\phi = e^{-\theta}$. OLS regression of $X_{t+1}$ on $X_t$ is therefore the **conditional MLE in closed form** -- no numerical optimizer needed. Recovering the OU parameters from the OLS fit $(\hat c, \hat\phi, \hat v)$ (intercept, slope, residual variance):

$$\hat\theta = -\ln\hat\phi \qquad \hat\mu = \hat c/(1-\hat\phi) \qquad \hat\sigma^2 = 2\hat\theta\hat v/(1-\hat\phi^2) \qquad \hat\tau = 1/\hat\theta$$

A valid mean-reverting fit requires $0 < \hat\phi < 1$ (equivalently $\hat\theta > 0$). $\hat\phi \geq 1$ is a unit root (random-walk-like, non-reverting); $\hat\phi \leq 0$ is an oscillating fit. Neither is a valid OU process -- these participants get NaN parameters (FLAG 4).


## Theory-to-data mapping

| Theoretical quantity | DS4 variable | Mapping type | Confidence |
|---|---|---|---|
| State variable X_t | `n_opened/n_events` per day, `intervention_event` class only | Proxy -- behavioral rate, not a latent state | Medium |
| OU drift mu | `mu_hat_OU` | Direct reuse | High for valid fits |
| Mean-reversion rate theta | `theta_hat_OU` from AR(1) OLS | Estimated | Medium -- real estimate; behavioral interpretation open |
| Diffusion sigma | `sigma_hat_OU` from AR(1) residual variance | Estimated | Medium-High |
| Relaxation time tau | `tau_hat = 1/theta_hat_OU` (days) | Direct transform | High as a number; interpretation flagged open |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

BEH_PATH = Path("outputs/cleaned/ds4_behavioral_clean.csv")
OUT_DIR = Path("outputs/model_outputs/ds4_stochastic_processes")
FIG_DIR = OUT_DIR / "figures"

MIN_DAYS = 30       # minimum daily X_t observations for daily OU eligibility
MIN_PAIRS = 10      # minimum consecutive-day pairs to attempt a fit
MIN_WEEKS = 5       # minimum weekly X_t observations for the weekly sensitivity check

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
beh = pd.read_csv(BEH_PATH, parse_dates=["timestamp_utc"], low_memory=False)
print(f"Loaded {len(beh):,} rows, {beh['participant_id_canonical'].nunique()} participants")

df = beh[beh["intervention_event_class"] == "intervention_event"].copy()
df = df[df["event_outcome_simple"].isin(["opened", "dismissed"])].copy()
df["opened_flag"] = (df["event_outcome_simple"] == "opened").astype(int)
df["event_date"] = df["timestamp_utc"].dt.date

daily = df.groupby(["participant_id_canonical", "event_date"]).agg(
    n_opened=("opened_flag", "sum"),
    n_events=("opened_flag", "count"),
).reset_index()
daily["X_t"] = daily["n_opened"] / daily["n_events"]
daily["event_date"] = pd.to_datetime(daily["event_date"])
daily = daily.sort_values(["participant_id_canonical", "event_date"]).reset_index(drop=True)

all_pids = beh["participant_id_canonical"].unique()
print(f"Daily X_t table: {len(daily):,} participant-day rows across {daily['participant_id_canonical'].nunique()} participants")


In [ ]:
def fit_ou_participant(g):
    """g: one participant's daily X_t rows, sorted by event_date."""
    n_days = len(g)
    gap = g["event_date"].diff().dt.days
    x_t = g["X_t"].values[:-1]
    x_t1 = g["X_t"].values[1:]
    consec = (gap.values[1:] == 1)
    x_t, x_t1 = x_t[consec], x_t1[consec]
    n_pairs = len(x_t)

    out = dict(n_days_obs=n_days, n_consec_pairs=n_pairs,
               phi_hat=np.nan, mu_hat_OU=np.nan, theta_hat_OU=np.nan,
               sigma_hat_OU=np.nan, tau_hat=np.nan, fit_valid=False)
    if n_days < MIN_DAYS or n_pairs < MIN_PAIRS:
        return out

    X = np.column_stack([np.ones_like(x_t), x_t])
    coef, *_ = np.linalg.lstsq(X, x_t1, rcond=None)
    c, phi = coef
    out["phi_hat"] = phi
    if not (0 < phi < 1):
        return out  # unit-root or oscillating fit -- not mean-reverting

    resid = x_t1 - X @ coef
    v = resid.var(ddof=2)
    theta = -np.log(phi)
    mu = c / (1 - phi)
    sigma2 = 2 * theta * v / (1 - phi ** 2)

    out.update(mu_hat_OU=mu, theta_hat_OU=theta,
                sigma_hat_OU=float(np.sqrt(max(sigma2, 0))),
                tau_hat=1 / theta, fit_valid=True)
    return out


daily_by_pid = {pid: g.sort_values("event_date").reset_index(drop=True)
                 for pid, g in daily.groupby("participant_id_canonical")}

records = []
for pid in all_pids:
    g = daily_by_pid.get(pid)
    if g is None or len(g) == 0:
        rec = dict(participant_id_canonical=pid, n_days_obs=0, n_consec_pairs=0,
                    phi_hat=np.nan, mu_hat_OU=np.nan, theta_hat_OU=np.nan,
                    sigma_hat_OU=np.nan, tau_hat=np.nan, fit_valid=False)
    else:
        rec = {"participant_id_canonical": pid, **fit_ou_participant(g)}
    records.append(rec)

participant_parameters = pd.DataFrame(records)
print(f"participant_parameters: {len(participant_parameters)} rows")
participant_parameters.head()


In [ ]:
n_total = len(all_pids)
n_any_day = int((participant_parameters["n_days_obs"] >= 1).sum())
n_min_days = int((participant_parameters["n_days_obs"] >= MIN_DAYS).sum())
n_min_pairs = int(((participant_parameters["n_days_obs"] >= MIN_DAYS) &
                    (participant_parameters["n_consec_pairs"] >= MIN_PAIRS)).sum())
n_valid = int(participant_parameters["fit_valid"].sum())

print("DS4 StoP eligibility funnel")
print(f"  Total participants:                       {n_total}")
print(f"  >=1 intervention-outcome day:              {n_any_day}")
print(f"  >=MIN_DAYS({MIN_DAYS}) daily observations:      {n_min_days}")
print(f"  ...and >=MIN_PAIRS({MIN_PAIRS}) consecutive pairs:  {n_min_pairs}")
if n_min_pairs > 0:
    print(f"  Valid mean-reverting fit (0<phi<1):       {n_valid} ({n_valid/n_min_pairs:.1%} of attempted)")


In [ ]:
valid = participant_parameters[participant_parameters["fit_valid"]].copy()
summary_cols = ["mu_hat_OU", "theta_hat_OU", "sigma_hat_OU", "tau_hat"]
dist_summary = valid[summary_cols].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
print(dist_summary)


In [ ]:
excluded_no_day = n_total - n_any_day
excluded_min_days = n_any_day - n_min_days
excluded_min_pairs = n_min_days - n_min_pairs
invalid_fits = n_min_pairs - n_valid

print("Exclusion breakdown:")
print(f"  No intervention-outcome day at all:  {excluded_no_day} ({excluded_no_day/n_total:.1%})")
print(f"  Below MIN_DAYS threshold:            {excluded_min_days} ({excluded_min_days/n_total:.1%})")
print(f"  Below MIN_PAIRS threshold:           {excluded_min_pairs} ({excluded_min_pairs/n_total:.1%})")
if n_min_pairs > 0:
    print(f"  Invalid fit (phi outside (0,1)):     {invalid_fits} ({invalid_fits/n_min_pairs:.1%} of attempted)")

attempted = participant_parameters[
    (participant_parameters["n_days_obs"] >= MIN_DAYS) &
    (participant_parameters["n_consec_pairs"] >= MIN_PAIRS)
]
fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(attempted["phi_hat"].dropna(), bins=40, color="#55A868", edgecolor="white")
ax.axvline(0, color="black", linestyle="--", linewidth=1)
ax.axvline(1, color="black", linestyle="--", linewidth=1)
ax.set_xlabel("phi_hat (AR(1) slope)")
ax.set_ylabel("count")
ax.set_title("phi_hat distribution -- valid mean-reverting range is (0,1)")
fig.tight_layout()
plt.show()


In [ ]:
eligible_pids = participant_parameters.loc[
    participant_parameters["n_days_obs"] >= MIN_DAYS, "participant_id_canonical"
]

def fit_theta_with_floor(g, min_events):
    g = g[g["n_events"] >= min_events].sort_values("event_date").reset_index(drop=True)
    gap = g["event_date"].diff().dt.days
    x_t = g["X_t"].values[:-1]
    x_t1 = g["X_t"].values[1:]
    consec = (gap.values[1:] == 1)
    x_t, x_t1 = x_t[consec], x_t1[consec]
    if len(x_t) < MIN_PAIRS:
        return np.nan
    X = np.column_stack([np.ones_like(x_t), x_t])
    coef, *_ = np.linalg.lstsq(X, x_t1, rcond=None)
    c, phi = coef
    if not (0 < phi < 1):
        return np.nan
    return -np.log(phi)

floor_summary = {}
for floor in [1, 3, 5]:
    thetas = []
    subset = daily[daily["participant_id_canonical"].isin(eligible_pids)]
    for pid, g in subset.groupby("participant_id_canonical"):
        theta = fit_theta_with_floor(g, floor)
        if not np.isnan(theta):
            thetas.append(theta)
    thetas = np.array(thetas)
    floor_summary[floor] = dict(n_valid=len(thetas),
                                  median_theta=float(np.median(thetas)),
                                  median_tau=float(np.median(1 / thetas)))
    print(f"min_events_per_day={floor}: n_valid={len(thetas)}, "
          f"median theta={np.median(thetas):.3f}, median tau={np.median(1/thetas):.2f} days")

floor_sensitivity_df = pd.DataFrame(floor_summary).T
floor_sensitivity_df.index.name = "min_events_per_day"


In [ ]:
participant_parameters.to_csv(OUT_DIR / "participant_parameters.csv", index=False)
print(f"Wrote participant_parameters.csv ({len(participant_parameters)} rows)")

summary_row = dict(
    n_total=n_total,
    n_any_day=n_any_day,
    n_min_days=n_min_days,
    n_min_pairs=n_min_pairs,
    n_valid_fit=n_valid,
    pct_valid_of_attempted=float(n_valid / n_min_pairs) if n_min_pairs else np.nan,
    median_mu_hat_OU=float(valid["mu_hat_OU"].median()),
    median_theta_hat_OU=float(valid["theta_hat_OU"].median()),
    median_sigma_hat_OU=float(valid["sigma_hat_OU"].median()),
    median_tau_hat=float(valid["tau_hat"].median()),
)
ds4_stop_summary = pd.DataFrame([summary_row])
ds4_stop_summary.to_csv(OUT_DIR / "ds4_stop_summary.csv", index=False)
print("Wrote ds4_stop_summary.csv")
print(ds4_stop_summary.T)

summary_lines = []
summary_lines.append("# DS4 StoP -- Model Run Summary")
summary_lines.append("")
summary_lines.append("**Notebook:** `53_ds4_stochastic_processes.ipynb`  ")
summary_lines.append("**Blueprint:** `outputs/reports/stage4/step1_modeling/ds4_stop_blueprint.md`  ")
summary_lines.append("**Run date:** " + pd.Timestamp.now().strftime("%Y-%m-%d"))
summary_lines.append("")
summary_lines.append("## Eligibility funnel")
summary_lines.append("")
summary_lines.append("| Stage | N |")
summary_lines.append("|---|---|")
summary_lines.append(f"| Total participants | {n_total} |")
summary_lines.append(f"| >=1 intervention-outcome day | {n_any_day} |")
summary_lines.append(f"| >=MIN_DAYS({MIN_DAYS}) daily observations | {n_min_days} |")
summary_lines.append(f"| ...and >=MIN_PAIRS({MIN_PAIRS}) consecutive pairs | {n_min_pairs} |")
summary_lines.append(f"| Valid mean-reverting fit (0<phi<1) | {n_valid} "
                     f"({n_valid/n_min_pairs:.1%} of attempted) |")
summary_lines.append("")
summary_lines.append(f"## Parameter distribution (valid fits only, N={n_valid})")
summary_lines.append("")
summary_lines.append("```")
summary_lines.append(dist_summary.to_string())
summary_lines.append("```")
summary_lines.append("")
summary_lines.append("## Event-count floor sensitivity (theta_hat_OU)")
summary_lines.append("")
summary_lines.append("```")
summary_lines.append(floor_sensitivity_df.to_string())
summary_lines.append("```")
summary_lines.append("")
summary_lines.append("## Headline finding")
summary_lines.append("")
summary_lines.append(f"Median relaxation time tau_hat = {valid['tau_hat'].median():.2f} "
                     "days. See blueprint Section 4 for the two candidate readings "
                     "(genuine fast habitual reversion vs. daily-ratio sampling noise) "
                     "and the weekly-aggregation sensitivity check below for which "
                     "reading the real output favors.")
summary_lines.append("")
summary_lines.append("## Weekly-aggregation sensitivity (daily vs weekly theta)")
summary_lines.append("")
summary_lines.append("See `participant_parameters_weekly_sensitivity.csv` and figure 4 "
                     "below.")

with open(OUT_DIR / "model_run_summary.md", "w") as f:
    f.write("\n".join(summary_lines) + "\n")
print("Wrote model_run_summary.md")


In [ ]:
df_w = df.copy()
df_w["event_week"] = df_w["timestamp_utc"].dt.to_period("W").apply(lambda p: p.start_time)

weekly = df_w.groupby(["participant_id_canonical", "event_week"]).agg(
    n_opened=("opened_flag", "sum"),
    n_events=("opened_flag", "count"),
).reset_index()
weekly["X_t"] = weekly["n_opened"] / weekly["n_events"]
weekly = weekly.rename(columns={"event_week": "event_date"})
weekly = weekly.sort_values(["participant_id_canonical", "event_date"]).reset_index(drop=True)

def fit_ou_weekly(g):
    n_periods = len(g)
    gap_weeks = g["event_date"].diff().dt.days / 7
    x_t = g["X_t"].values[:-1]
    x_t1 = g["X_t"].values[1:]
    consec = np.isclose(gap_weeks.values[1:], 1.0)
    x_t, x_t1 = x_t[consec], x_t1[consec]
    n_pairs = len(x_t)
    out = dict(n_weeks_obs=n_periods, n_consec_week_pairs=n_pairs,
               phi_hat_weekly=np.nan, theta_hat_OU_weekly=np.nan,
               tau_hat_weekly=np.nan, fit_valid_weekly=False)
    if n_periods < MIN_WEEKS or n_pairs < MIN_PAIRS:
        return out
    X = np.column_stack([np.ones_like(x_t), x_t])
    coef, *_ = np.linalg.lstsq(X, x_t1, rcond=None)
    c, phi = coef
    out["phi_hat_weekly"] = phi
    if not (0 < phi < 1):
        return out
    theta = -np.log(phi)
    out.update(theta_hat_OU_weekly=theta, tau_hat_weekly=1 / theta, fit_valid_weekly=True)
    return out

weekly_by_pid = {pid: g.sort_values("event_date").reset_index(drop=True)
                  for pid, g in weekly.groupby("participant_id_canonical")}

weekly_records = []
for pid in all_pids:
    g = weekly_by_pid.get(pid)
    if g is None or len(g) == 0:
        rec = dict(participant_id_canonical=pid, n_weeks_obs=0, n_consec_week_pairs=0,
                    phi_hat_weekly=np.nan, theta_hat_OU_weekly=np.nan,
                    tau_hat_weekly=np.nan, fit_valid_weekly=False)
    else:
        rec = {"participant_id_canonical": pid, **fit_ou_weekly(g)}
    weekly_records.append(rec)

participant_parameters_weekly = pd.DataFrame(weekly_records)
participant_parameters_weekly.to_csv(
    OUT_DIR / "participant_parameters_weekly_sensitivity.csv", index=False
)

n_valid_weekly = int(participant_parameters_weekly["fit_valid_weekly"].sum())
print(f"Weekly OU fit: {n_valid_weekly} valid mean-reverting fits")
if n_valid_weekly > 0:
    wv = participant_parameters_weekly[participant_parameters_weekly["fit_valid_weekly"]]
    print(f"Median weekly theta_hat_OU: {wv['theta_hat_OU_weekly'].median():.3f} (per week)")
    print(f"Median weekly tau_hat: {wv['tau_hat_weekly'].median():.2f} weeks")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(valid["theta_hat_OU"], bins=40, color="#4C72B0", edgecolor="white")
ax.set_xlabel("theta_hat_OU (mean-reversion rate, /day)")
ax.set_ylabel("count")
ax.set_title(f"DS4 StoP: theta_hat_OU distribution (n={len(valid)})")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig1_ds4_theta_distribution.png", dpi=150)
plt.close(fig)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
tau_vals = valid["tau_hat"]
bins = np.logspace(np.log10(tau_vals.min()), np.log10(tau_vals.max()), 40)
ax.hist(tau_vals, bins=bins, color="#C44E52", edgecolor="white")
ax.set_xscale("log")
ax.set_xlabel("tau_hat = 1/theta_hat_OU (days, log scale)")
ax.set_ylabel("count")
ax.set_title(f"DS4 StoP: relaxation time distribution (n={len(valid)})")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig2_ds4_tau_distribution.png", dpi=150)
plt.close(fig)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(valid["mu_hat_OU"], valid["sigma_hat_OU"], c=valid["tau_hat"],
                  cmap="viridis", s=18, alpha=0.8)
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label("tau_hat (days)")
ax.set_xlabel("mu_hat_OU (long-run mean override rate)")
ax.set_ylabel("sigma_hat_OU (diffusion)")
ax.set_title("DS4 StoP: mu vs sigma, colored by relaxation time")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig3_ds4_mu_sigma_scatter.png", dpi=150)
plt.close(fig)


In [ ]:
merged_theta = participant_parameters[
    ["participant_id_canonical", "theta_hat_OU", "fit_valid"]
].merge(
    participant_parameters_weekly[
        ["participant_id_canonical", "theta_hat_OU_weekly", "fit_valid_weekly"]
    ],
    on="participant_id_canonical", how="inner",
)
both_valid = merged_theta[merged_theta["fit_valid"] & merged_theta["fit_valid_weekly"]]

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(both_valid["theta_hat_OU"], both_valid["theta_hat_OU_weekly"],
            s=18, alpha=0.7, color="#8172B2")
ax.set_xlabel("theta_hat_OU (daily, /day)")
ax.set_ylabel("theta_hat_OU_weekly (weekly, /week)")
ax.set_title(f"Daily vs weekly theta_hat_OU (n={len(both_valid)} with both valid)")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig4_daily_vs_weekly_theta.png", dpi=150)
plt.close(fig)

print(f"Participants with valid daily AND weekly fit: {len(both_valid)}")
if len(both_valid) > 1:
    corr = both_valid["theta_hat_OU"].corr(both_valid["theta_hat_OU_weekly"])
    print(f"Correlation(daily theta, weekly theta): {corr:.3f}")


## Interpretation and limits

**Read the funnel and distribution cells above before citing any number from this notebook** -- the eligibility counts and parameter medians are computed live above, not hardcoded here.

**Fast mean-reversion finding:** the daily-timescale relaxation time is fast (see the tau_hat distribution and its median above). Two readings remain open, per the blueprint Section 4:

1. Genuine fast habitual reversion -- override behavior snaps back to a personal baseline within about a day.
2. Daily aggregation is a noisy ratio even at moderate event counts, and the event-count-floor sensitivity check (cell 12) only rules out *very* low counts, not sampling noise generally.

Compare the daily theta_hat_OU distribution (Figure 1) against the weekly one (Figure 4 / cell 14 print output). If weekly theta is dramatically slower than daily theta, reading 2 is favored; if it stays comparably fast, reading 1 gains support. This notebook reports both figures without picking a winner.

### DS4 weakness flags

- **FLAG I -- no experimental condition:** all results are within/cross-participant heterogeneity, never a friction effect.
- **FLAG II -- self-selected users:** the MIN_DAYS=30 eligible sample is not a random subsample of all 1,039 participants -- it is the subset who kept using OneSec long enough to be observed.
- **FLAG III -- fast-theta interpretation is open,** not resolved by this notebook (see above).
- **FLAG IV -- invalid-fit rate:** report the exclusion breakdown from cell 11 explicitly whenever quoting the eligible-N; do not conflate 'eligible for fitting' with 'has a valid fit'.
- **FLAG V -- behavioral proxy limitation:** X_t is a behavioral override rate, not a latent psychological state.


## Forward connections

| Connection | Mechanism | Target |
|---|---|---|
| DS4 StoP -> DS4 InT | sigma_hat_OU feeds the H_SDE Gaussian-entropy bridge | `52_ds4_information_theory.ipynb` |
| DS4 StoP -> DS4 DyS | theta_hat_OU/sigma_hat_OU partition regimes; OU gives a true interior fixed point X*=mu_hat_OU | `55_ds4_dynamical_systems.ipynb` |
| DS4 StoP -> DS4 StM | mu_hat_OU, theta_hat_OU, sigma_hat_OU feed the rigorous beta_OU=2*theta/sigma^2 Boltzmann bridge | `51_ds4_statistical_mechanics.ipynb` (blueprint already written) |
| DS4 StoP -> Paper | OU SDE, discrete AR(1) equivalence, theta/tau interpretation are not yet in the conceptual paper |

**Next step:** write the DS4 InT (52) blueprint, informed by this notebook's real executed sigma_hat_OU distribution -- not written in advance.
